In [ ]:
!pip install torch torchvision torchaudio 


In [ ]:

import os, math, time, random
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

import numpy as np
from PIL import Image, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

from diffusers import (
    UNet2DConditionModel,
    AutoencoderKL,
    DDPMScheduler,
    DPMSolverMultistepScheduler,
)
from transformers import CLIPTextModel, CLIPTokenizer

_HAS_SCIPY = False
try:
    from scipy.ndimage import distance_transform_edt
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| scipy:", _HAS_SCIPY)


In [ ]:

@dataclass
class Config:
    base_model: str = "runwayml/stable-diffusion-v1-5"

    # ROI + resolution
    image_size: int = 512
    latent_size: int = 64
    roi_margin_frac: float = 0.30
    min_roi_pixels: int = 24

    # Mask features
    use_mask_distance: bool = True
    mask_channels: int = 3  # auto-set below

    use_context_hint: bool = True
    context_blur_radius: float = 6.0
    ctx_drop_prob: float = 0.30  # dropout so mask-only still works

    # UNet input channels
    in_channels_latent: int = 4
    in_channels_ctx_latent: int = 4  # VAE latents for context hint
    in_channels_unet: int = 0        # auto-set below

    # Text conditioning
    prompt: str = "a realistic clinical photograph of a skin lesion"
    cfg_drop_prob: float = 0.10  # drop text conditioning sometimes

    # Training
    lr: float = 1e-5
    weight_decay: float = 1e-2
    batch_size: int = 2
    grad_accum: int = 8
    num_workers: int = 4
    max_steps: int = 25000
    mixed_precision: bool = True

    # Min-SNR weighting
    use_min_snr: bool = True
    min_snr_gamma: float = 5.0

    # EMA
    use_ema: bool = True
    ema_decay: float = 0.9999

    # Sampling
    infer_steps: int = 30
    guidance_scale: float = 5.0
    seed: int = 42

    # Output
    output_dir: str = "wg2diff_roi_v3"
    save_ema_weights: bool = True


cfg = Config()

# Adjust mask channels based on scipy availability
if cfg.use_mask_distance and _HAS_SCIPY:
    cfg.mask_channels = 3  # binary, dist_in, dist_out
else:
    cfg.mask_channels = 1  # binary only

# UNet in_channels: noisy_latents(4) + mask_lat(C) + ctx_latents(4 if enabled else 0)
cfg.in_channels_unet = cfg.in_channels_latent + cfg.mask_channels + (cfg.in_channels_ctx_latent if cfg.use_context_hint else 0)

print("mask_channels:", cfg.mask_channels, "| in_channels_unet:", cfg.in_channels_unet)


In [ ]:

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

def mask_bbox(mask_np: np.ndarray) -> Optional[Tuple[int,int,int,int]]:
    ys, xs = np.where(mask_np > 0.5)
    if len(xs) == 0:
        return None
    x0, x1 = xs.min(), xs.max() + 1
    y0, y1 = ys.min(), ys.max() + 1
    return x0, y0, x1, y1

def expand_bbox(x0,y0,x1,y1, w,h, margin_frac: float):
    bw = x1 - x0
    bh = y1 - y0
    mx = int(bw * margin_frac)
    my = int(bh * margin_frac)
    nx0 = max(0, x0 - mx)
    ny0 = max(0, y0 - my)
    nx1 = min(w, x1 + mx)
    ny1 = min(h, y1 + my)
    return nx0, ny0, nx1, ny1

def compute_snr_weights(scheduler: DDPMScheduler, timesteps: torch.Tensor, gamma: float) -> torch.Tensor:
    alphas_cumprod = scheduler.alphas_cumprod.to(device=timesteps.device)
    a = alphas_cumprod[timesteps]  # [B]
    snr = a / (1 - a)
    w = torch.minimum(snr, torch.full_like(snr, gamma)) / snr
    return w

def pil_blur(img: Image.Image, radius: float) -> Image.Image:
    if radius <= 0:
        return img
    return img.filter(ImageFilter.GaussianBlur(radius=radius))


In [ ]:

class ROIWoundDatasetV3(Dataset):
    """
    Returns:
      pixel_values: [3,512,512] in [-1,1]  (real crop)
      mask_feats:   [C,512,512] in [0,1]
      ctx_pixels:   [3,512,512] in [-1,1]  (blurred crop; low-freq context hint)
    """
    def __init__(self, image_paths: List[str], mask_paths: List[str], cfg: Config):
        assert len(image_paths) == len(mask_paths)
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.cfg = cfg

    def __len__(self):
        return len(self.image_paths)

    def _load_rgb(self, path: str) -> Image.Image:
        return Image.open(path).convert("RGB")

    def _load_mask(self, path: str) -> Image.Image:
        return Image.open(path).convert("L")

    def _make_mask_feats(self, mask_01: np.ndarray) -> np.ndarray:
        bin_mask = (mask_01 > 0.5).astype(np.float32)
        if self.cfg.mask_channels == 1:
            return bin_mask[None, ...]
        # distance features
        if not _HAS_SCIPY:
            return bin_mask[None, ...]
        inside = bin_mask.astype(bool)
        outside = ~inside
        dist_in = distance_transform_edt(inside).astype(np.float32)
        dist_out = distance_transform_edt(outside).astype(np.float32)
        eps = 1e-6
        dist_in = dist_in / (dist_in.max() + eps)
        dist_out = dist_out / (dist_out.max() + eps)
        feats = np.stack([bin_mask, dist_in, dist_out], axis=0)
        return feats

    def __getitem__(self, idx):
        img = self._load_rgb(self.image_paths[idx])
        msk = self._load_mask(self.mask_paths[idx])

        img_np = np.array(img).astype(np.float32) / 255.0
        msk_np = np.array(msk).astype(np.float32) / 255.0

        h, w = msk_np.shape[:2]
        bb = mask_bbox(msk_np)
        if bb is None:
            return {"skip": True}

        x0,y0,x1,y1 = bb
        if (x1 - x0) < self.cfg.min_roi_pixels or (y1 - y0) < self.cfg.min_roi_pixels:
            return {"skip": True}

        x0,y0,x1,y1 = expand_bbox(x0,y0,x1,y1, w=w, h=h, margin_frac=self.cfg.roi_margin_frac)

        # Crop
        img_crop = img_np[y0:y1, x0:x1, :]
        msk_crop = msk_np[y0:y1, x0:x1]

        # Resize -> 512
        img_crop_pil = Image.fromarray((img_crop * 255).astype(np.uint8)).resize(
            (self.cfg.image_size, self.cfg.image_size), Image.BICUBIC
        )
        msk_crop_pil = Image.fromarray((msk_crop * 255).astype(np.uint8)).resize(
            (self.cfg.image_size, self.cfg.image_size), Image.NEAREST
        )

        # Context hint = blurred version of the same crop
        ctx_pil = pil_blur(img_crop_pil, radius=self.cfg.context_blur_radius) if self.cfg.use_context_hint else img_crop_pil

        img_arr = np.array(img_crop_pil).astype(np.float32) / 255.0
        ctx_arr = np.array(ctx_pil).astype(np.float32) / 255.0
        msk_arr = np.array(msk_crop_pil).astype(np.float32) / 255.0

        # Normalize images to [-1,1]
        img_arr = (img_arr * 2.0) - 1.0
        ctx_arr = (ctx_arr * 2.0) - 1.0

        mask_feats = self._make_mask_feats(msk_arr)  # [C,512,512]

        pixel_values = torch.from_numpy(img_arr).permute(2,0,1).contiguous()
        ctx_pixels   = torch.from_numpy(ctx_arr).permute(2,0,1).contiguous()
        mask_feats   = torch.from_numpy(mask_feats).contiguous()

        return {"pixel_values": pixel_values, "ctx_pixels": ctx_pixels, "mask_feats": mask_feats, "skip": False}

def collate_skip_v3(batch):
    batch = [b for b in batch if not b.get("skip", False)]
    if len(batch) == 0:
        return None
    pixel_values = torch.stack([b["pixel_values"] for b in batch], dim=0)
    ctx_pixels   = torch.stack([b["ctx_pixels"] for b in batch], dim=0)
    mask_feats   = torch.stack([b["mask_feats"] for b in batch], dim=0)
    return {"pixel_values": pixel_values, "ctx_pixels": ctx_pixels, "mask_feats": mask_feats}


In [ ]:

class MaskEncoderLite(nn.Module):
    """
    ControlNet-lite: takes mask_lat [B, Cmask, 64, 64]
    outputs:
      down4: list of 4 residuals with channels matching SD1.5 UNet:
        64x64 -> 320
        32x32 -> 640
        16x16 -> 1280
        8x8  -> 1280
      mid:   [B,1280,8,8]
    """
    def __init__(self, c_mask: int):
        super().__init__()

        # A small conv trunk
        self.stem = nn.Sequential(
            nn.Conv2d(c_mask, 64, 3, padding=1),
            nn.SiLU(),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.SiLU(),
        )

        # Downsample stages to create 64/32/16/8 scales
        self.to64 = nn.Sequential(
            nn.Conv2d(128, 128, 3, padding=1),
            nn.SiLU(),
        )
        self.to32 = nn.Sequential(
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.SiLU(),
        )
        self.to16 = nn.Sequential(
            nn.Conv2d(256, 512, 3, stride=2, padding=1),
            nn.SiLU(),
        )
        self.to8 = nn.Sequential(
            nn.Conv2d(512, 512, 3, stride=2, padding=1),
            nn.SiLU(),
        )

        # 1x1 projections to EXACT UNet channel sizes (this fixes your error)
        self.proj64 = nn.Conv2d(128, 320, 1)
        self.proj32 = nn.Conv2d(256, 640, 1)
        self.proj16 = nn.Conv2d(512, 1280, 1)
        self.proj8  = nn.Conv2d(512, 1280, 1)

        # Mid residual is same as 8x8 in SD1.5
        self.proj_mid = nn.Conv2d(512, 1280, 1)

        # Small init so it doesn't overpower early training
        for m in [self.proj64, self.proj32, self.proj16, self.proj8, self.proj_mid]:
            nn.init.normal_(m.weight, mean=0.0, std=1e-3)
            nn.init.zeros_(m.bias)

    def forward(self, mask_lat):
        x = self.stem(mask_lat)      # [B,128,64,64]
        f64 = self.to64(x)           # [B,128,64,64]
        f32 = self.to32(f64)         # [B,256,32,32]
        f16 = self.to16(f32)         # [B,512,16,16]
        f8  = self.to8(f16)          # [B,512,8,8]

        r64 = self.proj64(f64)       # [B,320,64,64]
        r32 = self.proj32(f32)       # [B,640,32,32]
        r16 = self.proj16(f16)       # [B,1280,16,16]
        r8  = self.proj8(f8)         # [B,1280,8,8]
        mid = self.proj_mid(f8)      # [B,1280,8,8]

        down4 = [r64, r32, r16, r8]
        return down4, mid


In [ ]:

class WG2DiffROI_V3(nn.Module):
    def __init__(self, cfg: Config, device: torch.device):
        super().__init__()
        self.cfg = cfg
        self.device = device

        # Frozen components
        self.vae = AutoencoderKL.from_pretrained(cfg.base_model, subfolder="vae").to(device)
        self.text_encoder = CLIPTextModel.from_pretrained(cfg.base_model, subfolder="text_encoder").to(device)
        self.tokenizer = CLIPTokenizer.from_pretrained(cfg.base_model, subfolder="tokenizer")

        # Trainable UNet
        self.unet = UNet2DConditionModel.from_pretrained(cfg.base_model, subfolder="unet").to(device)

        # Scheduler
        self.train_scheduler = DDPMScheduler.from_pretrained(cfg.base_model, subfolder="scheduler")

        # Expand conv_in: 4 (noisy latents) + maskC + 4 (context latents)
        old = self.unet.conv_in
        new_conv = nn.Conv2d(
            in_channels=cfg.in_channels_unet,
            out_channels=old.out_channels,
            kernel_size=old.kernel_size,
            stride=old.stride,
            padding=old.padding,
        ).to(device)

        # Initialize: copy first 4 channels (base SD behavior), small init for new channels
        with torch.no_grad():
            new_conv.weight.zero_()
            new_conv.bias.copy_(old.bias)
            new_conv.weight[:, :cfg.in_channels_latent].copy_(old.weight)
            if cfg.in_channels_unet > cfg.in_channels_latent:
                nn.init.normal_(new_conv.weight[:, cfg.in_channels_latent:], mean=0.0, std=1e-3)

        self.unet.conv_in = new_conv

        # ControlNet-lite mask encoder (multi-scale injection)
        block_out = list(self.unet.config.block_out_channels)
        self.mask_encoder = MaskEncoderLite(c_mask=cfg.mask_channels).to(device)

        # Freeze VAE + text
        self.vae.requires_grad_(False)
        self.text_encoder.requires_grad_(False)

        self.latent_scaling = 0.18215

    @torch.no_grad()
    def encode_text(self, prompt: str) -> torch.Tensor:
        tokens = self.tokenizer(
            prompt,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        )
        return self.text_encoder(tokens.input_ids.to(self.device)).last_hidden_state

    @torch.no_grad()
    def encode_image_to_latents(self, pixel_values: torch.Tensor) -> torch.Tensor:
        posterior = self.vae.encode(pixel_values).latent_dist
        latents = posterior.sample() * self.latent_scaling
        return latents

    @torch.no_grad()
    def decode_latents_to_image(self, latents: torch.Tensor) -> torch.Tensor:
        latents = latents / self.latent_scaling
        img = self.vae.decode(latents).sample
        img = (img / 2 + 0.5).clamp(0, 1)
        return img

    def maskfeats_to_latent(self, mask_feats_512: torch.Tensor) -> torch.Tensor:
        # [B,C,512,512] -> [B,C,64,64]
        return F.interpolate(mask_feats_512, size=(self.cfg.latent_size, self.cfg.latent_size), mode="nearest")


In [ ]:

class EMA:
    def __init__(self, model: nn.Module, decay: float):
        self.decay = decay
        self.shadow = {}
        self.collected = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = p.detach().clone()

    @torch.no_grad()
    def update(self, model: nn.Module):
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name].mul_(self.decay).add_(p.detach(), alpha=(1.0 - self.decay))

    def store(self, model: nn.Module):
        self.collected = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.collected[name] = p.detach().clone()

    def copy_to(self, model: nn.Module):
        for name, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.shadow[name].data)

    def restore(self, model: nn.Module):
        for name, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.collected[name].data)
        self.collected = {}


In [ ]:

def train_v3(
    image_paths: List[str],
    mask_paths: List[str],
    cfg: Config,
    device: torch.device,
):
    set_seed(cfg.seed)

    dataset = ROIWoundDatasetV3(image_paths, mask_paths, cfg)
    loader = DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
        collate_fn=collate_skip_v3,
        drop_last=True,
    )

    model = WG2DiffROI_V3(cfg, device).to(device)
    model.train()

    # Train UNet + mask_encoder
    train_params = list(model.unet.parameters()) + list(model.mask_encoder.parameters())
    optimizer = torch.optim.AdamW(train_params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda" and cfg.mixed_precision))

    ema = EMA(model.unet, decay=cfg.ema_decay) if cfg.use_ema else None

    # Precompute prompt embeddings (cond/uncond)
    with torch.no_grad():
        text_cond = model.encode_text(cfg.prompt)  # [1,77,768]
        text_uncond = model.encode_text("")        # [1,77,768]

    def expand_down_residuals_to_sd15(down4, target_len=12):
        """
        SD1.5 UNet typically expects 12 down residual tensors:
          [64x64]*3 + [32x32]*3 + [16x16]*3 + [8x8]*3

        down4: list of 4 tensors at roughly [64,32,16,8] (order may vary).
        Returns list length 12 in correct spatial order.
        """
        assert isinstance(down4, (list, tuple)) and len(down4) == 4, f"Expected 4 residuals, got {type(down4)} len={len(down4) if isinstance(down4,(list,tuple)) else 'NA'}"

        # Sort by spatial size descending to enforce [64,32,16,8]
        down4_sorted = sorted(down4, key=lambda x: x.shape[-1], reverse=True)

        out = []
        for t in down4_sorted:
            out.extend([t, t, t])  # repeat 3x per resolution

        # Adjust length if a diffusers variant expects a different number
        if len(out) > target_len:
            out = out[:target_len]
        elif len(out) < target_len:
            out.extend([out[-1]] * (target_len - len(out)))

        return out

    def train_one_batch(batch: Dict) -> torch.Tensor:
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)  # real crop [-1,1]
        ctx_pixels   = batch["ctx_pixels"].to(device, non_blocking=True)    # blurred crop [-1,1]
        mask_feats_512 = batch["mask_feats"].to(device, non_blocking=True)

        bsz = pixel_values.shape[0]

        # Mask -> latent
        mask_lat = model.maskfeats_to_latent(mask_feats_512)  # [B,Cmask,64,64]

        # Encode real image -> latents (target)
        with torch.no_grad():
            latents = model.encode_image_to_latents(pixel_values)

        # Encode context hint -> latents, then dropout sometimes (so model can be mask-only)
        if cfg.use_context_hint:
            with torch.no_grad():
                ctx_latents = model.encode_image_to_latents(ctx_pixels)  # [B,4,64,64]
            drop = (torch.rand(bsz, device=device) < cfg.ctx_drop_prob).float().view(bsz, 1, 1, 1)
            ctx_latents = ctx_latents * (1.0 - drop)  # dropped samples become zeros
        else:
            ctx_latents = None

        # CFG text dropout per-sample
        use_cond = (torch.rand(bsz, device=device) > cfg.cfg_drop_prob)
        enc = torch.where(
            use_cond[:, None, None],
            text_cond.repeat(bsz, 1, 1),
            text_uncond.repeat(bsz, 1, 1),
        )

        # Sample noise level
        timesteps = torch.randint(
            0, model.train_scheduler.config.num_train_timesteps, (bsz,), device=device, dtype=torch.long
        )

        # Add noise
        noise = torch.randn_like(latents)
        noisy_latents = model.train_scheduler.add_noise(latents, noise, timesteps)

        # UNet input = noisy_latents + mask_lat (+ ctx_latents)
        if cfg.use_context_hint:
            unet_in = torch.cat([noisy_latents, mask_lat, ctx_latents], dim=1)
        else:
            unet_in = torch.cat([noisy_latents, mask_lat], dim=1)

        # Multi-scale mask residuals (ControlNet-lite)
        # IMPORTANT FIX: expand 4-scale residuals -> SD1.5 expected residual list
       
        with torch.cuda.amp.autocast(enabled=(cfg.mixed_precision and device.type == "cuda"), dtype=torch.float16):
            out = model.unet(
                unet_in,
                timesteps,
                encoder_hidden_states=enc,
               
            )
            noise_pred = out.sample

            mse = (noise_pred - noise) ** 2
            mse = mse.mean(dim=(1,2,3))  # per-sample

            if cfg.use_min_snr:
                w = compute_snr_weights(model.train_scheduler, timesteps, cfg.min_snr_gamma)
                loss = (w * mse).mean()
            else:
                loss = mse.mean()

        return loss

    os.makedirs(cfg.output_dir, exist_ok=True)

    global_step = 0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(total=cfg.max_steps, desc="Training v3", dynamic_ncols=True)

    while global_step < cfg.max_steps:
        for batch in loader:
            if batch is None:
                continue

            loss = train_one_batch(batch)
            loss = loss / cfg.grad_accum
            scaler.scale(loss).backward()

            if (global_step + 1) % cfg.grad_accum == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if ema is not None:
                    ema.update(model.unet)

            global_step += 1
            pbar.update(1)

            if global_step >= cfg.max_steps:
                break

    pbar.close()

    # Save final weights
    final_dir = os.path.join(cfg.output_dir, "final")
    os.makedirs(final_dir, exist_ok=True)

    # Save UNet (optionally EMA)
    unet_dir = os.path.join(final_dir, "unet")
    os.makedirs(unet_dir, exist_ok=True)

    if ema is not None and cfg.save_ema_weights:
        ema.store(model.unet)
        ema.copy_to(model.unet)
        model.unet.save_pretrained(unet_dir, safe_serialization=True)
        ema.restore(model.unet)
    else:
        model.unet.save_pretrained(unet_dir, safe_serialization=True)

    # Save mask encoder
    torch.save(model.mask_encoder.state_dict(), os.path.join(final_dir, "mask_encoder.pt"))

    # Save config
    with open(os.path.join(cfg.output_dir, "train_config.txt"), "w") as f:
        for k, v in cfg.__dict__.items():
            f.write(f"{k}: {v}\n")

    return model, ema


In [ ]:

@torch.no_grad()
def build_mask_feats_from_pil(mask_pil: Image.Image, cfg: Config) -> torch.Tensor:
    msk = (np.array(mask_pil).astype(np.float32) / 255.0)
    bin_mask = (msk > 0.5).astype(np.float32)

    if cfg.mask_channels == 1 or not _HAS_SCIPY:
        feats = bin_mask[None, ...]
    else:
        inside = bin_mask.astype(bool)
        outside = ~inside
        dist_in = distance_transform_edt(inside).astype(np.float32)
        dist_out = distance_transform_edt(outside).astype(np.float32)
        eps = 1e-6
        dist_in = dist_in / (dist_in.max() + eps)
        dist_out = dist_out / (dist_out.max() + eps)
        feats = np.stack([bin_mask, dist_in, dist_out], axis=0)

    return torch.from_numpy(feats)[None, ...]  # [1,C,512,512]

@torch.no_grad()
def generate_roi_v3(
    model: WG2DiffROI_V3,
    ema: Optional[EMA],
    mask_feats_512: torch.Tensor,            # [1,C,512,512]
    prompt: str,
    steps: int,
    guidance: float,
    seed: Optional[int] = None,
    ctx_pixels_512: Optional[torch.Tensor] = None,  # [1,3,512,512] in [-1,1] (optional)
    use_ema: bool = True,
):
    model.eval()
    if seed is not None:
        set_seed(seed)

    sampler = DPMSolverMultistepScheduler.from_config(model.train_scheduler.config)
    sampler.set_timesteps(steps, device=device)

    if ema is not None and use_ema:
        ema.store(model.unet)
        ema.copy_to(model.unet)

    cond = model.encode_text(prompt)
    uncond = model.encode_text("")
    enc = torch.cat([uncond, cond], dim=0)  # [2,77,768]

    mask_feats_512 = mask_feats_512.to(device)
    mask_lat = model.maskfeats_to_latent(mask_feats_512)  # [1,C,64,64]

    # Context latents (optional)
    if model.cfg.use_context_hint:
        if ctx_pixels_512 is None:
            ctx_lat = torch.zeros((1, 4, model.cfg.latent_size, model.cfg.latent_size), device=device)
        else:
            ctx_pixels_512 = ctx_pixels_512.to(device)
            ctx_lat = model.encode_image_to_latents(ctx_pixels_512)
    else:
        ctx_lat = None

    # Mask residuals
    down_residuals, mid_residual = model.mask_encoder(mask_lat)

    latents = torch.randn((1, 4, model.cfg.latent_size, model.cfg.latent_size), device=device)

    for t in sampler.timesteps:
        latent_in = torch.cat([latents, latents], dim=0)
        mask_in = mask_lat.repeat(2, 1, 1, 1)

        if model.cfg.use_context_hint:
            ctx_in = ctx_lat.repeat(2, 1, 1, 1)
            unet_in = torch.cat([latent_in, mask_in, ctx_in], dim=1)
        else:
            unet_in = torch.cat([latent_in, mask_in], dim=1)

        noise_pred = model.unet(
            unet_in,
            t,
            encoder_hidden_states=enc,
            down_block_additional_residuals=[r.repeat(2,1,1,1) for r in down_residuals],
            mid_block_additional_residual=mid_residual.repeat(2,1,1,1),
        ).sample

        n_u, n_c = noise_pred.chunk(2)
        noise = n_u + guidance * (n_c - n_u)
        latents = sampler.step(noise, t, latents).prev_sample

    img = model.decode_latents_to_image(latents)  # [1,3,512,512]

    if ema is not None and use_ema:
        ema.restore(model.unet)

    return img


In [ ]:
import os, math, time
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

def save_sample_grid(img_tensor_bchw, out_path):
    """
    img_tensor_bchw: [B,3,512,512] in [0,1]
    Saves first image only for simplicity.
    """
    from PIL import Image
    import numpy as np
    x = img_tensor_bchw[0].detach().cpu().clamp(0,1)
    x = (x.permute(1,2,0).numpy() * 255).astype("uint8")
    Image.fromarray(x).save(out_path)

@torch.no_grad()
def quick_sample(model, ema, cfg, device, batch, step_tag=""):
    """
    Generates one sample from the first mask in the batch.
    Uses mask-only sampling by default (ctx dropped to zeros).
    """
    model.eval()

    # Use EMA for sampling if available
    if ema is not None and cfg.use_ema:
        ema.store(model.unet)
        ema.copy_to(model.unet)

    # Prepare conditioning
    mask_feats_512 = batch["mask_feats"][:1].to(device)
    mask_lat = model.maskfeats_to_latent(mask_feats_512)

    # context latents: mask-only mode by default
    if cfg.use_context_hint:
        ctx_lat = torch.zeros((1, 4, cfg.latent_size, cfg.latent_size), device=device)
    else:
        ctx_lat = None

    # Text embeddings (cond/uncond)
    cond = model.encode_text(cfg.prompt)
    uncond = model.encode_text("")
    enc = torch.cat([uncond, cond], dim=0)

    # Residuals from mask encoder
    down_residuals, mid_residual = model.mask_encoder(mask_lat)

    # Sampler
    sampler = DPMSolverMultistepScheduler.from_config(model.train_scheduler.config)
    sampler.set_timesteps(cfg.infer_steps, device=device)

    latents = torch.randn((1, 4, cfg.latent_size, cfg.latent_size), device=device)

    for t in sampler.timesteps:
        latent_in = torch.cat([latents, latents], dim=0)
        mask_in = mask_lat.repeat(2, 1, 1, 1)

        if cfg.use_context_hint:
            ctx_in = ctx_lat.repeat(2, 1, 1, 1)
            unet_in = torch.cat([latent_in, mask_in, ctx_in], dim=1)
        else:
            unet_in = torch.cat([latent_in, mask_in], dim=1)

        noise_pred = model.unet(
            unet_in,
            t,
            encoder_hidden_states=enc,
            down_block_additional_residuals=[r.repeat(2,1,1,1) for r in down_residuals],
            mid_block_additional_residual=mid_residual.repeat(2,1,1,1),
        ).sample

        n_u, n_c = noise_pred.chunk(2)
        noise = n_u + cfg.guidance_scale * (n_c - n_u)
        latents = sampler.step(noise, t, latents).prev_sample

    img = model.decode_latents_to_image(latents)  # [1,3,512,512] in [0,1]

    if ema is not None and cfg.use_ema:
        ema.restore(model.unet)

    model.train()
    return img

def train_v3_loop(image_paths, mask_paths, cfg, device):
    set_seed(cfg.seed)
    os.makedirs(cfg.output_dir, exist_ok=True)
    sample_dir = os.path.join(cfg.output_dir, "samples")
    os.makedirs(sample_dir, exist_ok=True)

    # Dataset + loader
    dataset = ROIWoundDatasetV3(image_paths, mask_paths, cfg)
    loader = DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
        collate_fn=collate_skip_v3,
        drop_last=True,
    )

    # Model
    model = WG2DiffROI_V3(cfg, device).to(device)
    model.train()

    # Optimizer: UNet + mask_encoder
    params = list(model.unet.parameters()) + list(model.mask_encoder.parameters())
    optimizer = torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)

    # AMP
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda" and cfg.mixed_precision))

    # EMA
    ema = EMA(model.unet, decay=cfg.ema_decay) if cfg.use_ema else None

    # Precompute text embeddings (cond/uncond) once
    with torch.no_grad():
        text_cond = model.encode_text(cfg.prompt)  # [1,77,768]
        text_uncond = model.encode_text("")        # [1,77,768]

    global_step = 0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(total=cfg.max_steps, desc="Training v3", dynamic_ncols=True)

    while global_step < cfg.max_steps:
        for batch in loader:
            if batch is None:
                continue

            pixel_values = batch["pixel_values"].to(device, non_blocking=True)  # real crop [-1,1]
            ctx_pixels   = batch["ctx_pixels"].to(device, non_blocking=True)    # blurred crop [-1,1]
            mask_feats_512 = batch["mask_feats"].to(device, non_blocking=True)

            bsz = pixel_values.shape[0]

            # Mask -> latent (for concat + residuals)
            mask_lat = model.maskfeats_to_latent(mask_feats_512)  # [B,Cmask,64,64]

            # Encode real -> latents (target)
            with torch.no_grad():
                latents = model.encode_image_to_latents(pixel_values)

            # Encode context hint -> latents, then dropout per-sample
            if cfg.use_context_hint:
                with torch.no_grad():
                    ctx_latents = model.encode_image_to_latents(ctx_pixels)  # [B,4,64,64]
                drop = (torch.rand(bsz, device=device) < cfg.ctx_drop_prob).float().view(bsz,1,1,1)
                ctx_latents = ctx_latents * (1.0 - drop)  # dropped -> zeros
            else:
                ctx_latents = None

            # CFG dropout for text conditioning per-sample
            use_cond = (torch.rand(bsz, device=device) > cfg.cfg_drop_prob)
            enc = torch.where(
                use_cond[:, None, None],
                text_cond.repeat(bsz, 1, 1),
                text_uncond.repeat(bsz, 1, 1),
            )

            # Noise level
            timesteps = torch.randint(
                0, model.train_scheduler.config.num_train_timesteps, (bsz,),
                device=device, dtype=torch.long
            )

            # Forward diffusion
            noise = torch.randn_like(latents)
            noisy_latents = model.train_scheduler.add_noise(latents, noise, timesteps)

            # UNet input concat
            if cfg.use_context_hint:
                unet_in = torch.cat([noisy_latents, mask_lat, ctx_latents], dim=1)
            else:
                unet_in = torch.cat([noisy_latents, mask_lat], dim=1)

            # Multi-scale mask residuals (ControlNet-lite)
            down_residuals, mid_residual = model.mask_encoder(mask_lat)

            with torch.cuda.amp.autocast(
                enabled=(cfg.mixed_precision and device.type == "cuda"),
                dtype=torch.float16
            ):
                noise_pred = model.unet(
                    unet_in,
                    timesteps,
                    encoder_hidden_states=enc,
                    down_block_additional_residuals=down_residuals,
                    mid_block_additional_residual=mid_residual,
                ).sample

                mse = (noise_pred - noise) ** 2
                mse = mse.mean(dim=(1,2,3))  # [B]

                if cfg.use_min_snr:
                    w = compute_snr_weights(model.train_scheduler, timesteps, cfg.min_snr_gamma)
                    loss = (w * mse).mean()
                else:
                    loss = mse.mean()

            # Backprop (grad accumulation)
            loss = loss / cfg.grad_accum
            scaler.scale(loss).backward()

            if (global_step + 1) % cfg.grad_accum == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if ema is not None:
                    ema.update(model.unet)

            global_step += 1
            pbar.update(1)
            pbar.set_postfix({"loss": float(loss.detach().cpu())})

            # Periodic sampling
            if global_step % 2000 == 0:
                try:
                    sample = quick_sample(model, ema, cfg, device, batch, step_tag=str(global_step))
                    save_sample_grid(sample, os.path.join(sample_dir, f"step_{global_step:06d}.png"))
                except Exception as e:
                    print("⚠️ sampling failed:", e)

            
            if global_step >= cfg.max_steps:
                break

        if global_step >= cfg.max_steps:
            break

    pbar.close()

    # Final save
    final_dir = os.path.join(cfg.output_dir, "final")
    os.makedirs(final_dir, exist_ok=True)

    unet_dir = os.path.join(final_dir, "unet")
    os.makedirs(unet_dir, exist_ok=True)

    if ema is not None and cfg.save_ema_weights:
        ema.store(model.unet)
        ema.copy_to(model.unet)
        model.unet.save_pretrained(unet_dir, safe_serialization=True)
        ema.restore(model.unet)
    else:
        model.unet.save_pretrained(unet_dir, safe_serialization=True)

    torch.save(model.mask_encoder.state_dict(), os.path.join(final_dir, "mask_encoder.pt"))

    with open(os.path.join(cfg.output_dir, "train_config.txt"), "w") as f:
        for k, v in cfg.__dict__.items():
            f.write(f"{k}: {v}\n")

    print("✅ done. Saved to:", cfg.output_dir)
    return model, ema


In [ ]:
import torch.nn.functional as F

def expand_down_residuals_to_sd15(down4, target_len=12):
    """
    down4: list/tuple of 4 tensors at spatial sizes [64, 32, 16, 8] (or close).
    Returns a list of length 12 matching SD1.5 down_block_res_samples layout:
      [64x64]*3 + [32x32]*3 + [16x16]*3 + [8x8]*3
    If the input order is wrong, we reorder by spatial size.
    """
    assert len(down4) == 4, f"Expected 4 residual maps, got {len(down4)}"

    down4_sorted = sorted(down4, key=lambda x: x.shape[-1], reverse=True)

    sizes = [t.shape[-1] for t in down4_sorted]  # e.g. [64,32,16,8]

    out = []
    for t in down4_sorted:
        out.extend([t, t, t])

    if len(out) != target_len:
        if len(out) > target_len:
            out = out[:target_len]
        else:
            out.extend([out[-1]] * (target_len - len(out)))

    return out


In [ ]:
import os, glob

IMAGE_DIRS = [
    "/home/sagemaker-user/final_dataset/final_dataset/AZH",
    "/home/sagemaker-user/final_dataset/final_dataset/Kaggle_images",
    "/home/sagemaker-user/final_dataset/final_dataset/WoundDB",
]
MASK_DIRS = [
    "/home/sagemaker-user/final_dataset/final_dataset/AZH_masks",
    "/home/sagemaker-user/final_dataset/final_dataset/Kaggle_Masks",
    "/home/sagemaker-user/final_dataset/final_dataset/WoundDB_masks",
]

def collect_pairs(image_dirs, mask_dirs, exts_img=("jpg","jpeg","png"), exts_msk=("png","jpg","jpeg")):
    img_map = {}
    for d in image_dirs:
        for ext in exts_img:
            for p in glob.glob(os.path.join(d, f"*.{ext}")):
                stem = os.path.splitext(os.path.basename(p))[0]
                img_map[stem] = p

    pairs = []
    for d in mask_dirs:
        for ext in exts_msk:
            for m in glob.glob(os.path.join(d, f"*.{ext}")):
                stem = os.path.splitext(os.path.basename(m))[0]
                if stem in img_map:
                    pairs.append((img_map[stem], m))

    pairs = sorted(pairs)
    image_paths = [p[0] for p in pairs]
    mask_paths  = [p[1] for p in pairs]
    return image_paths, mask_paths

image_paths, mask_paths = collect_pairs(IMAGE_DIRS, MASK_DIRS)

print("Pairs found:", len(image_paths))
print("Example pair:\n ", image_paths[0] if image_paths else None, "\n ", mask_paths[0] if mask_paths else None)

assert len(image_paths) > 0, "No pairs found — check folder paths and naming."
assert len(image_paths) == len(mask_paths)

ds = ROIWoundDatasetV3(image_paths, mask_paths, cfg)
x = None
for i in range(min(50, len(ds))):
    sample = ds[i]
    if not sample.get("skip", False):
        x = sample
        break

assert x is not None, "Dataset is skipping everything (likely masks empty / wrong pairing / wrong threshold)."
print("✅ dataset sample ok:", x["pixel_values"].shape, x["mask_feats"].shape, x["ctx_pixels"].shape)

# ---- 4) START TRAINING ----
model, ema = train_v3(image_paths, mask_paths, cfg, device)


In [ ]:
# Image Generation 
@torch.no_grad()
def generate_v3_image(
    model,
    mask_path: str,
    cfg,
    device,
    seed: Optional[int] = None,
):
    model.eval()

    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    expected_in = model.unet.conv_in.in_channels  
    latent_c = 4
    extra_c = expected_in - latent_c
    if extra_c <= 0:
        raise ValueError(f"UNet conv_in expects {expected_in} channels; expected >4 for conditioning.")

    
    uses_ctx = False
    if extra_c >= 5:  
        uses_ctx = bool(getattr(cfg, "use_context_hint", False)) and (extra_c >= 4 + 1)

    mask_lat_c = extra_c - (4 if uses_ctx else 0)
    if mask_lat_c <= 0:
        raise ValueError(f"Computed mask_lat channels {mask_lat_c} is invalid. expected_in={expected_in}, uses_ctx={uses_ctx}")

    # Load mask (full -> ROI -> 512) 
    mask_pil = Image.open(mask_path).convert("L")
    mask_np = np.array(mask_pil).astype(np.float32) / 255.0

    bb = mask_bbox(mask_np)
    if bb is None:
        raise ValueError("Mask has no lesion pixels")

    x0, y0, x1, y1 = expand_bbox(
        *bb,
        w=mask_np.shape[1],
        h=mask_np.shape[0],
        margin_frac=cfg.roi_margin_frac,
    )

    mask_crop = mask_np[y0:y1, x0:x1]
    mask_crop = Image.fromarray((mask_crop * 255).astype(np.uint8)).resize(
        (cfg.image_size, cfg.image_size),
        Image.NEAREST,
    )
    mask_arr = np.array(mask_crop).astype(np.float32) / 255.0

    # Build mask features to match mask_lat_c 
    if hasattr(model, "build_mask_feats_from_array"):
        feats = model.build_mask_feats_from_array(mask_arr)  # expected [C,512,512]
    else:
        bin_mask = (mask_arr > 0.5).astype(np.float32)
        feats_list = [bin_mask]

        if mask_lat_c > 1:
            if "_HAS_SCIPY" in globals() and _HAS_SCIPY:
                inside = bin_mask.astype(bool)
                outside = ~inside
                dist_in = distance_transform_edt(inside).astype(np.float32)
                dist_out = distance_transform_edt(outside).astype(np.float32)
                eps = 1e-6
                dist_in = dist_in / (dist_in.max() + eps)
                dist_out = dist_out / (dist_out.max() + eps)
                feats_list = [bin_mask, dist_in, dist_out]
            while len(feats_list) < mask_lat_c:
                feats_list.append(feats_list[-1])
            feats_list = feats_list[:mask_lat_c]

        feats = np.stack(feats_list, axis=0)

    mask_feats_512 = torch.from_numpy(feats)[None].to(device)  # [1,Cmask,512,512]
    mask_lat = model.maskfeats_to_latent(mask_feats_512)       # [1,Cmask,64,64]

    if uses_ctx:
        ctx_lat = torch.zeros((1, 4, cfg.latent_size, cfg.latent_size), device=device)
    else:
        ctx_lat = None

    # Text embeddings 
    cond = model.encode_text(cfg.prompt)
    uncond = model.encode_text("")
    enc = torch.cat([uncond, cond], dim=0)

    # Sampler 
    sampler = DPMSolverMultistepScheduler.from_config(model.train_scheduler.config)
    sampler.set_timesteps(cfg.infer_steps, device=device)

    latents = torch.randn((1, 4, cfg.latent_size, cfg.latent_size), device=device)

    #  Diffusion loop 
    for t in sampler.timesteps:
        latent_in = torch.cat([latents, latents], dim=0)           # [2,4,64,64]
        mask_in = mask_lat.repeat(2, 1, 1, 1)                      # [2,Cmask,64,64]

        if uses_ctx:
            ctx_in = ctx_lat.repeat(2, 1, 1, 1)                   # [2,4,64,64]
            unet_in = torch.cat([latent_in, mask_in, ctx_in], dim=1)
        else:
            unet_in = torch.cat([latent_in, mask_in], dim=1)

        if unet_in.shape[1] != expected_in:
            raise RuntimeError(f"UNet expects {expected_in} channels but got {unet_in.shape[1]}")

        noise_pred = model.unet(unet_in, t, encoder_hidden_states=enc).sample
        noise_uncond, noise_cond = noise_pred.chunk(2)
        noise = noise_uncond + cfg.guidance_scale * (noise_cond - noise_uncond)

        latents = sampler.step(noise, t, latents).prev_sample

    #  Decode 
    img = model.decode_latents_to_image(latents)
    img = (img[0].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    return Image.fromarray(img)
